In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — Stock prices

**New concept: `pd.merge_asof()`**

`merge_asof` is a left join that matches on the *nearest* key rather than an exact key. It's the right tool when one table has events at irregular times and you want to look up the most recent value from another table.

```python
pd.merge_asof(left, right, on='timestamp')
```

**Rules:**
- Both DataFrames must be **sorted** by the key column before merging.
- Default `direction='backward'`: for each row in `left`, find the largest key in `right` that is ≤ the left key (i.e., the most recent past value).
- `direction='forward'`: find the smallest key in `right` that is ≥ the left key.
- `direction='nearest'`: find the closest key in either direction.
- `by='col'`: match exactly on this column first (like a group key), then do the asof match within each group.

```python
# prices updated at irregular intervals; trades happen at their own times
# find the prevailing price at each trade time:
pd.merge_asof(trades, prices, on='time')   # both must be sorted by 'time'
```

`prices` has a price recorded at certain timestamps. `trades` records a purchase at a different set of times. For each trade, find the most recent price.

1. Sort both DataFrames by `time` (they may already be sorted — check).
2. Use `merge_asof` to attach the prevailing price to each trade.
3. Add a `trade_value = quantity * price` column. Which trade had the highest value? Use `np.argmax`.
4. Use a named-agg groupby on the result: group by `side` and compute total quantity and total trade value.

In [3]:
prices = pd.DataFrame({
    'time':  pd.to_datetime(['09:30','09:45','10:00','10:15','10:30','10:45','11:00'], format='%H:%M'),
    'price': [100.0, 101.5, 99.8, 102.3, 103.1, 101.9, 104.5],
})

trades = pd.DataFrame({
    'time':     pd.to_datetime(['09:32','09:52','10:08','10:22','10:38','10:51'], format='%H:%M'),
    'quantity': [50, 30, 100, 75, 20, 60],
    'side':     ['buy', 'sell', 'buy', 'buy', 'sell', 'buy'],
})

# Your code here
prices = prices.sort_values('time')
trades = trades.sort_values('time')

m = pd.merge_asof(
    trades,
    prices, 
    on = 'time'
)

m['trade_value'] = m['quantity']* m['price']

print(m.iloc[np.argmax(m['trade_value'])],'has the largest trade_value')


m.groupby('side').agg(

    total_quantity = ('quantity','sum'),
    total_trade_value = ('trade_value','sum')
)

time           1900-01-01 10:08:00
quantity                       100
side                           buy
price                         99.8
trade_value                 9980.0
Name: 2, dtype: object has the largest trade_value


,total_quantity,total_trade_value
side,,
buy,285,28766.5
sell,50,5107.0


---

## Level 2 — Wage history

`wage_changes` logs every time an employee's hourly wage was updated. `timesheets` records hours worked on specific dates. For each timesheet entry, find the wage that was in effect on that date.

Use `merge_asof` with `by='employee_id'` — this matches exactly on employee first, then does the nearest-date lookup within each employee's records.

1. Merge the two tables. Add `pay = hours * wage`.
2. What was the total pay earned by each employee? Use named agg on the merged result.
3. Which employee had the highest average pay per shift? Use `np.argmax`.
4. Collect the employee IDs into a plain Python list using a list comprehension.

In [14]:
wage_changes = pd.DataFrame({
    'date':        pd.to_datetime(['2023-01-01','2023-06-01','2024-01-01',
                                   '2023-01-01','2023-09-01','2024-03-01']),
    'employee_id': ['E01','E01','E01','E02','E02','E02'],
    'wage':        [18.0, 20.0, 22.0, 15.0, 17.5, 19.0],
}).sort_values('date')

timesheets = pd.DataFrame({
    'date':        pd.to_datetime(['2023-03-10','2023-07-22','2024-02-14',
                                   '2023-05-05','2023-11-30','2024-04-01']),
    'employee_id': ['E01','E01','E01','E02','E02','E02'],
    'hours':       [8, 7.5, 8, 9, 8, 7],
}).sort_values('date')

# Your code here

m = pd.merge_asof(
    timesheets,
    wage_changes, 
    on = 'date',
    by = 'employee_id'
)

m['pay'] = m['hours']*m['wage']

m.groupby('employee_id').agg(
    total_pay = ('pay','sum')
)
mp = m.groupby('employee_id').agg(
    avg_pay = ('pay','mean')
)
print(mp['avg_pay'].idxmax(),'has the highest average pay')
[id for id in m['employee_id']]


E01 has the highest average pay


['E01', 'E02', 'E01', 'E02', 'E01', 'E02']

---

## Level 3 — Monthly sales

**New concept: `pd.merge_ordered()`**

`merge_ordered` is like an outer merge that preserves sort order. Its main use is merging two time series that have different dates — the result spans the full union of dates, with NaN where one series has no data.

```python
pd.merge_ordered(left, right, on='month')
```

The key extra parameter is `fill_method='ffill'` — this forward-fills NaN values after the merge, carrying the last known value forward:

```python
pd.merge_ordered(left, right, on='month', fill_method='ffill')
# NaN in the result is replaced by the previous non-NaN value
```

`product_a` and `product_b` report revenue in alternating months. Merge them into a single monthly timeline.

1. Use `merge_ordered` with `fill_method='ffill'` to create a complete monthly view. Drop any rows still NaN after the fill (the very first rows may have no prior value to fill from).
2. Add `total = revenue_a + revenue_b`. In which month was combined revenue highest?
3. Use `np.corrcoef` to check whether the two products' revenues move together. Print the value.
4. Use a list comprehension to collect the months where `total > 23000`.

In [34]:
product_a = pd.DataFrame({
    'month':     pd.to_datetime(['2023-01','2023-03','2023-05','2023-07','2023-09','2023-11'], format='%Y-%m'),
    'revenue_a': [11200, 12800, 10900, 13500, 12100, 14300],
})

product_b = pd.DataFrame({
    'month':     pd.to_datetime(['2023-02','2023-04','2023-06','2023-08','2023-10','2023-12'], format='%Y-%m'),
    'revenue_b': [9400, 10200, 9800, 11500, 10600, 12200],
})

# Your code here


f = pd.merge_ordered(product_a, product_b, on = 'month', fill_method='ffill')
f['total'] = f['revenue_a']+f['revenue_b']
print(f.loc[f['total'].idxmax(),'month'],'has the hihest combined revenue')
fi = f.dropna()
print(np.corrcoef(fi['revenue_a'], fi['revenue_b'])[0,1])
print('they move together')

#[m for m in fi['month'] if fi.loc[fi['month']==m,'total']>23000]
[m for m,t in zip (fi['month'], fi['total']) if t >23000]

2023-12-01 00:00:00 has the hihest combined revenue
0.49668207389284647
they move together


[Timestamp('2023-07-01 00:00:00'),
 Timestamp('2023-08-01 00:00:00'),
 Timestamp('2023-09-01 00:00:00'),
 Timestamp('2023-11-01 00:00:00'),
 Timestamp('2023-12-01 00:00:00')]

In [33]:
[m for m in fi['month'] if fi.loc[fi['month']==m,'total'].values[0]>23000]


[Timestamp('2023-07-01 00:00:00'),
 Timestamp('2023-08-01 00:00:00'),
 Timestamp('2023-09-01 00:00:00'),
 Timestamp('2023-11-01 00:00:00'),
 Timestamp('2023-12-01 00:00:00')]